# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PART 1: KHÁM PHÁ DỮ LIỆU & LÀM SẠCH VĂN BẢN (DATA CLEANING & EDA)

```text
[CEAS_08.csv] ──> Mục 1 (EDA Nền tảng: Cấu trúc, Chất lượng & Đơn biến)
              ──> Mục 2 (EDA Chuyên sâu: Đa biến, Ngoại lai & Đúc kết định hướng)
              ──> Mục 3 (Làm sạch dữ liệu: Khử trùng, điền missing, clean_text) ──> [clean_emails.csv]
```


---
## Mục 1: Khám phá dữ liệu tổng quan — Cấu trúc, Chất lượng & Đơn biến (Overview EDA)


In [ ]:
# Khai báo thư viện & Thiết lập môi trường
import os
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

DATA_RAW_PATH = '../data/raw/CEAS_08.csv'
print(f'✅ Môi trường sẵn sàng với Python {sys.version.split()[0]} | RANDOM_SEED = {RANDOM_SEED}')

In [ ]:
# Khảo sát cấu trúc & Đánh giá chất lượng dữ liệu
df = pd.read_csv("../data/raw/CEAS_08.csv")
print(f"📊 Kích thước dữ liệu gốc: {df.shape[0]:,} dòng, {df.shape[1]} cột")
print("-" * 60)
print("Thông tin kiểu dữ liệu và bộ nhớ:")
df.info()

print("\nNăm dòng đầu tiên của dữ liệu:")
display(df.head(5))

print("\nThống kê mô tả các biến số:")
display(df.describe())

# Đánh giá giá trị khuyết thiếu (Missing values)
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({
    'Số lượng thiếu': missing_count,
    'Tỷ lệ thiếu (%)': missing_percent.round(4)
})
print("\nThống kê giá trị khuyết thiếu trên các cột:")
display(missing_summary[missing_summary['Số lượng thiếu'] > 0])

# Đánh giá bản ghi trùng lặp (Duplicate records)
duplicate_count = df.duplicated().sum()
print(f"\n• Số dòng trùng lặp hoàn toàn: {duplicate_count:,} dòng ({(duplicate_count/len(df))*100:.2f}%)")


In [ ]:
# Phân tích đơn biến 4 thuộc tính
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Biến mục tiêu: label (0: Ham, 1: Spam)
label_counts = df['label'].value_counts()
sns.countplot(data=df, x='label', palette=['#3498db', '#e74c3c'], ax=axes[0, 0])
axes[0, 0].set_title(f"1. Biến mục tiêu: Ham ({label_counts.get(0, 0):,}) vs Spam ({label_counts.get(1, 0):,})")
axes[0, 0].set_xlabel('Nhãn (0: Ham / Hợp lệ, 1: Spam / Thư rác)')
axes[0, 0].set_ylabel('Số lượng email')
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f"{int(p.get_height()):,} ({p.get_height()/len(df)*100:.1f}%)",
                         (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                         ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# Biến số độc lập: urls (Có/Không link URL)
sns.countplot(data=df, x='urls', palette='viridis', ax=axes[0, 1])
axes[0, 1].set_title('2. Biến số URLs: Tỷ lệ email có chứa liên kết')
axes[0, 1].set_xlabel('Số lượng liên kết URL (0: Không, 1+: Có)')
axes[0, 1].set_ylabel('Số lượng email')

# Biến văn bản: subject length (Độ dài tiêu đề)
df['subject_len'] = df['subject'].fillna('').astype(str).apply(len)
sns.histplot(df['subject_len'], bins=50, kde=True, color='#2ecc71', ax=axes[1, 0])
axes[1, 0].set_title('3. Phân phối độ dài tiêu đề (Subject Length)')
axes[1, 0].set_xlabel('Số ký tự trong tiêu đề')
axes[1, 0].set_ylabel('Mật độ / Số lượng')
axes[1, 0].set_xlim(0, 200)

# Biến danh mục: sender domain (Tên miền người gửi)
df['sender_domain'] = df['sender'].astype(str).apply(lambda x: x.split('@')[-1].lower() if '@' in x else 'unknown')
top_domains = df['sender_domain'].value_counts().head(10)
top_domains.plot(kind='bar', color='#f39c12', ax=axes[1, 1])
axes[1, 1].set_title('4. Top 10 tên miền người gửi phổ biến nhất')
axes[1, 1].set_xlabel('Tên miền (Domain)')
axes[1, 1].set_ylabel('Số lượng email')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


---
## Mục 2: Khám phá dữ liệu chuyên sâu — Đa biến, Ngoại lai & Đúc kết định hướng (In-depth EDA)


In [ ]:
# Vẽ biểu đồ so sánh độ dài văn bản theo nhãn
# Tạo đặc trưng độ dài
df['subject_len'] = df['subject'].fillna('').astype(str).str.len()
df['body_len'] = df['body'].fillna('').astype(str).str.len()

# Vẽ biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df,
    x='label',
    y='subject_len',
    hue='label',
    legend=False,
    ax=axes[0]
)
axes[0].set_title('Độ dài tiêu đề theo nhãn Spam/Ham')
axes[0].set_xlabel('Nhãn (0: Ham, 1: Spam)')
axes[0].set_ylabel('Số ký tự')

sns.boxplot(
    data=df,
    x='label',
    y='body_len',
    hue='label',
    legend=False,
    ax=axes[1]
)
axes[1].set_title('Độ dài nội dung theo nhãn Spam/Ham')
axes[1].set_xlabel('Nhãn (0: Ham, 1: Spam)')
axes[1].set_ylabel('Số ký tự')

plt.tight_layout()
plt.show()


In [ ]:
# Độ dài tiêu đề và nội dung email
df['subject_len'] = df['subject'].fillna('').astype(str).str.len()
df['body_len'] = df['body'].fillna('').astype(str).str.len()

# Kiểm tra thống kê độ dài theo nhãn
display(
    df.groupby('label')[['subject_len', 'body_len']]
      .describe()
      .round(2)
)

In [ ]:
# Tạo bảng phát hiện ngoại lai bằng IQR
def detect_outliers_iqr(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_mask = (
        (data[column] < lower) |
        (data[column] > upper)
    )

    return {
        'Thuộc tính': column,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'Ngưỡng dưới': lower,
        'Ngưỡng trên': upper,
        'Số ngoại lai': int(outlier_mask.sum()),
        'Tỷ lệ ngoại lai (%)':
            round(outlier_mask.mean() * 100, 2)
    }


outlier_results = pd.DataFrame([
    detect_outliers_iqr(df, 'subject_len'),
    detect_outliers_iqr(df, 'body_len')
])

display(outlier_results.round(2))

In [ ]:
# Kiểm tra mối quan hệ giữa sự xuất hiện của liên kết URLs và nhãn Spam/Ham
url_crosstab = pd.crosstab(df['urls'], df['label'], normalize='index') * 100
print('📊 Tỷ lệ % Spam/Ham theo sự xuất hiện của URL:')
display(url_crosstab.round(2))


In [ ]:
# Bảng đúc kết kết luận EDA định hướng hành động
eda_conclusions = pd.DataFrame([
    {
        "Bằng chứng EDA": "28 email khuyết tiêu đề subject",
        "Ý nghĩa / Tác động": "Gây lỗi khi ghép chuỗi hoặc vector hóa văn bản",
        "Định hướng xử lý": "Mục 3: Điền chuỗi rỗng '' trước khi gộp text"
    },
    {
        "Bằng chứng EDA": "Độ dài nội dung (body) lệch phải nặng (đuôi dài)",
        "Ý nghĩa / Tác động": "Ngoại lai IQR làm sai lệch trọng số các thuật toán tuyến tính",
        "Định hướng xử lý": "Mục 4 (Notebook 2): Biến đổi Log np.log1p & Binning độ dài"
    },
    {
        "Bằng chứng EDA": "Email chứa liên kết URLs có tỷ lệ Spam cao vượt trội",
        "Ý nghĩa / Tác động": "Là đặc trưng phân biệt quan trọng nhưng chưa đủ bao quát",
        "Định hướng xử lý": "Mục 4 (Notebook 2): Tạo thêm biến Domain (caps_ratio, !, $)"
    },
    {
        "Bằng chứng EDA": "Số lượng từ vựng văn bản thô quá lớn",
        "Ý nghĩa / Tác động": "Gây bùng nổ số chiều (Curse of Dimensionality) và quá khớp",
        "Định hướng xử lý": "Mục 5 (Notebook 2): Trích xuất đặc trưng văn bản TF-IDF và chọn lọc đặc trưng bằng Chi-square"
    }
])

print("📋 BẢNG ĐÚC KẾT KẾT LUẬN & ĐỊNH HƯỚNG HÀNH ĐỘNG EDA (BƯỚC 5):")
display(eda_conclusions)

---
## Mục 3: Tiền xử lý dữ liệu & Làm sạch văn bản (Data Cleaning)


In [ ]:
# Khử trùng lặp hoàn toàn và điền chuỗi rỗng cho email thiếu subject
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Đã loại bỏ {rows_before - len(df):,} dòng trùng lặp ({rows_before:,} -> {len(df):,} dòng)")

print(f"Số email thiếu subject trước khi điền: {df['subject'].isnull().sum()}")
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')
print(f"Số email thiếu subject sau khi điền: {df['subject'].isnull().sum()}")

In [ ]:
# Xây dựng hàm clean_text, làm sạch subject và body, gộp thành text_clean
import re
import html
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Bộ stop words tiếng Anh tiêu chuẩn có sẵn từ scikit-learn (chạy 100% offline)
STOP_WORDS = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    tokens = [t for t in text.split() if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)

print(clean_text("<p>FREE!!! Click http://spam.com to WIN $1000 now</p>"))

df['subject_clean'] = df['subject'].apply(clean_text)
df['body_clean'] = df['body'].apply(clean_text)
df['text_clean'] = (df['subject_clean'] + ' ' + df['body_clean']).str.strip()

display(df[['subject', 'subject_clean', 'text_clean']].head())
print(f"Số dòng text_clean rỗng: {(df['text_clean'] == '').sum()}")


In [ ]:
# Xuất tệp dữ liệu sạch
import os
os.makedirs('../data/processed', exist_ok=True)

cols_to_save = ['sender', 'receiver', 'date', 'subject', 'body', 'text_clean', 'urls', 'label']
df[cols_to_save].to_csv('../data/processed/clean_emails.csv', index=False, encoding='utf-8')

print(f"Đã lưu clean_emails.csv: {len(df):,} dòng, {len(cols_to_save)} cột")
